In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q kaggle

In [ ]:
# Step 1: Kaggle credentials ko sahi folder mein set karna
!mkdir -p /root/.kaggle
!cp /content/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

# Step 2: Connection check karna (Agar ye chal gaya toh credentials sahi hain)
print("--- Testing Kaggle Connection ---")
!kaggle datasets list -s "knee"

# Step 3: Competition files ko inspect karna
print("\n--- Inspecting Competition Files ---")
!kaggle competitions files -c rsna-knee-abnormality-detection


cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
--- Testing Kaggle Connection ---
ref                                                        title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
mohamedgobara/osteoporosis-database                        Knee Osteoporosis Dataset multiclasses                190537183  2023-10-25 16:51:48.230000           2076         34                1  
shashwatwork/knee-osteoarthritis-dataset-with-severity     Knee Osteoarthritis Dataset with Severity Grading     213819666  2021-04-08 06:50:09.450000          19546        129            0.875  
dreaddevelopment/raptor-knee-widedense  

## kaggle API key setup

In [ ]:
import os

# Google Drive ke andar dataset ka destination path set karna
drive_destination = '/content/drive/MyDrive/Kaggle_Knee_Data'

if not os.path.exists(drive_destination):
    os.makedirs(drive_destination)
    print(f"Folder successfully ban gaya hai yahan: {drive_destination} ✅")
else:
    print("Folder pehle se hi bana hua hai! 👍")


Folder pehle se hi bana hua hai! 👍


In [ ]:
# 4. Target destination folder create karein
!mkdir -p /content/drive/MyDrive/Kaggle_Knee_Data

# 5. Download official competition DICOM zip directly
!kaggle competitions download -c rsna-knee-abnormality-detection -p /content/drive/MyDrive/Kaggle_Knee_Data

401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


In [ ]:
# Original DICOM stacks uploaded as dataset (No Competition API block issue)
!kaggle datasets download -d dreaddevelopment/knee-raptor-corpus -p /content/drive/MyDrive/Kaggle_Knee_Data --unzip

Dataset URL: https://www.kaggle.com/datasets/dreaddevelopment/knee-raptor-corpus
License(s): CC0-1.0
100% 11.1G/11.1G [19:19<00:00, 10.3MB/s]



In [ ]:
import os

data_dir = '/content/drive/MyDrive/Kaggle_Knee_Data'
train_series_path = os.path.join(data_dir, 'train_series')
test_series_path = os.path.join(data_dir, 'test_series')

# Train series count verify karein
if os.path.exists(train_series_path):
    train_folders = os.listdir(train_series_path)
    print(f"✅ Total MRI Series Folders in train_series: {len(train_folders)}")

    # Ek sample series folder check karein
    if len(train_folders) > 0:
        sample_folder = os.path.join(train_series_path, train_folders[0])
        images = os.listdir(sample_folder) if os.path.isdir(sample_folder) else []
        print(f"📸 Sample folder '{train_folders[0]}' contains {len(images)} images/slices.")

# Test series count verify karein
if os.path.exists(test_series_path):
    test_folders = os.listdir(test_series_path)
    print(f"✅ Total MRI Series Folders in test_series: {len(test_folders)}")

✅ Total MRI Series Folders in train_series: 153
📸 Sample folder '1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260' contains 5 images/slices.
✅ Total MRI Series Folders in test_series: 3


In [ ]:
import os
import glob

data_dir = '/content/drive/MyDrive/Kaggle_Knee_Data'

# Search for file extensions in the directory
dcm_files = glob.glob(f"{data_dir}/**/*.dcm", recursive=True)
jpg_files = glob.glob(f"{data_dir}/**/*.jpg", recursive=True) + glob.glob(f"{data_dir}/**/*.jpeg", recursive=True)
npy_files = glob.glob(f"{data_dir}/**/*.npy", recursive=True)

print(f"📁 DICOM (.dcm) files found: {len(dcm_files)}")
print(f"🖼️ JPEG (.jpg/.jpeg) files found: {len(jpg_files)}")
print(f"🔢 NumPy (.npy) array files found: {len(npy_files)}")

📁 DICOM (.dcm) files found: 0
🖼️ JPEG (.jpg/.jpeg) files found: 29525
🔢 NumPy (.npy) array files found: 4


In [ ]:
!pip install iterative-stratification

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Paths Definition
DATA_DIR = '/content/drive/MyDrive/Kaggle_Knee_Data'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, 'train_series.csv'))

# 12 Target Abnormality Columns
target_cols = [c for c in train_df.columns if c not in ['StudyInstanceUID']]

# Grouping by Patient ID to guarantee 0% Data Leakage
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
study_df = train_df.groupby('StudyInstanceUID')[target_cols].max().reset_index()

study_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(mskf.split(study_df['StudyInstanceUID'], study_df[target_cols].values)):
    study_df.loc[val_idx, 'fold'] = fold_idx

fold_map = dict(zip(study_df['StudyInstanceUID'], study_df['fold']))
train_df['fold'] = train_df['StudyInstanceUID'].map(fold_map)

# Save split to Drive
train_df.to_csv(os.path.join(DATA_DIR, 'train_folds.csv'), index=False)
print("✅ Grouped 5-Fold Stratified Split Ready!")

ValueError: Input contains NaN

## most of the work flow

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from typing import Dict, Optional
from functools import lru_cache
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as F
import gc

class KneeAbnormalityDatasetV2(Dataset):
    """
    Improved 3D Volumetric Dataset for Knee MRI
    - Patient-level grouping support
    - Better error handling
    - Medical-specific augmentations
    - Caching for faster loading
    """

    def __init__(
        self,
        df: pd.DataFrame,
        series_df: pd.DataFrame,
        data_dir: str,
        depth: int = 32,
        target_cols: Optional[list] = None,
        is_train: bool = True,
        cache_size: int = 1000,  # Kitne patients ka data cache karein
        patient_col: str = 'StudyInstanceUID'  # Patient identifier
    ):
        """
        Args:
            df: Main dataframe with labels
            series_df: Series metadata (StudyInstanceUID -> SeriesInstanceUID mapping)
            data_dir: Root directory containing MRI images
            depth: Number of slices to sample (default 32)
            target_cols: List of label columns (ACL, MCL, Meniscus, etc.)
            is_train: True for training set (applies augmentations)
            cache_size: LRU cache size for faster repeated access
            patient_col: Column name for patient ID (for grouping)
        """

        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.data_dir = data_dir
        self.depth = depth
        self.target_cols = target_cols or []
        self.is_train = is_train
        self.patient_col = patient_col

        # Patient-level mapping (Grouping ke liye zaroori)
        self.patient_to_idx = {
            pid: self.df[self.df[patient_col] == pid].index.tolist()
            for pid in self.df[patient_col].unique()
        }

        # Image Augmentations - Train vs Validation alag-alag
        if self.is_train:
            self.transform = self._get_train_transforms()
        else:
            self.transform = self._get_val_transforms()

        # Statistics tracking (Debugging ke liye)
        self.stats = {
            'empty_series': 0,
            'corrupt_images': 0,
            'total_loaded': 0
        }

    def _get_train_transforms(self):
        """Training augmentations - Medical images ke liye optimized"""
        return T.Compose([
            T.Resize((224, 224)),
            # Medical-specific augmentations
            T.RandomHorizontalFlip(p=0.5),  # Left-right knee symmetry
            T.RandomRotation(degrees=15),   # Slight rotation acceptable
            T.RandomAffine(
                degrees=10,
                translate=(0.05, 0.05),     # 5% translation max
                scale=(0.95, 1.05)          # 5% scaling variation
            ),
            T.ColorJitter(
                brightness=0.1,             # Thoda brightness change
                contrast=0.1                # Contrast change
            ),
            T.GaussianBlur(kernel_size=3),  # Simulate MRI noise
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def _get_val_transforms(self):
        """Validation transforms - No random augmentations"""
        return T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @lru_cache(maxsize=1000)
    def _load_3d_stack_cached(self, series_id: str):
        """
        Cached version of 3D stack loading
        - Same patient baar-baar load nahi hoga
        - 5-10x faster repeated access
        """
        return self._load_3d_stack_core(series_id)

    def _load_3d_stack_core(self, series_id: str):
        """
        Core logic for loading 3D MRI volume
        Handles: Empty series, corrupt images, variable slice counts
        """
        series_path = os.path.join(self.data_dir, 'train_series', str(series_id))

        # Support for multiple image formats
        img_paths = (
            sorted(glob.glob(os.path.join(series_path, "*.jpg"))) +
            sorted(glob.glob(os.path.join(series_path, "*.jpeg"))) +
            sorted(glob.glob(os.path.join(series_path, "*.png")))
        )

        # Fallback for empty/missing series
        if len(img_paths) == 0:
            self.stats['empty_series'] += 1
            print(f"️  Warning: Empty series {series_id}")
            return torch.zeros((3, self.depth, 224, 224))

        # Uniform Z-axis sampling (32 slices)
        indices = np.linspace(0, len(img_paths) - 1, self.depth).astype(int)
        selected_paths = [img_paths[i] for i in indices]

        slices = []
        for path in selected_paths:
            try:
                img = Image.open(path).convert('RGB')
                transformed = self.transform(img)
                slices.append(transformed)
            except Exception as e:
                # Corrupt image handling
                self.stats['corrupt_images'] += 1
                print(f"⚠️  Corrupt image: {path} - {str(e)}")
                # Create blank slice as fallback
                slices.append(torch.zeros((3, 224, 224)))

        # Stack into 3D volume: (3, 32, 224, 224)
        volume = torch.stack(slices, dim=1)
        self.stats['total_loaded'] += 1

        return volume

    def _load_3d_stack(self, series_id: str):
        """Wrapper that uses cache if available"""
        if hasattr(self, '_load_3d_stack_cached'):
            return self._load_3d_stack_cached(series_id)
        return self._load_3d_stack_core(series_id)

    def __len__(self):
        return len(self.df)

    def get_patient_indices(self, patient_id: str):
        """
        Returns all indices for a given patient
        Useful for grouped cross-validation
        """
        return self.patient_to_idx.get(patient_id, [])

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Returns a single sample with:
        - image: 3D volume (3, 32, 224, 224)
        - label: Multi-label targets
        - mask: Valid label mask (for -1 values)
        - study_id: For tracking/debugging
        """
        row = self.df.iloc[idx]
        study_id = row['StudyInstanceUID']

        # Get all series for this study (ek patient ki multiple series ho sakti hain)
        s_rows = self.series_df[self.series_df['StudyInstanceUID'] == study_id]

        if len(s_rows) == 0:
            print(f"⚠️  No series found for study {study_id}")
            series_id = ""
        else:
            # Strategy: Use first series (ya phir sabko merge kar sakte hain)
            series_id = s_rows.iloc[0]['SeriesInstanceUID']

            # Optional: Agar multiple series hain, toh sabko load karke average kar sakte hain
            if len(s_rows) > 1:
                print(f"️  Study {study_id} has {len(s_rows)} series, using first one")

        # Load 3D volume
        volume = self._load_3d_stack(series_id)

        # Extract labels
        labels = row[self.target_cols].values.astype(np.float32) if self.target_cols else np.array([])

        # Handle missing labels (-1 = not mentioned in report)
        mask = (labels != -1).astype(np.float32)
        clean_labels = np.where(labels == -1, 0.0, labels)

        return {
            'image': volume,                                    # (3, 32, 224, 224)
            'label': torch.tensor(clean_labels, dtype=torch.float32),
            'mask': torch.tensor(mask, dtype=torch.float32),
            'study_id': study_id,                               # Tracking ke liye
            'series_id': series_id
        }

    def get_statistics(self):
        """Dataset loading statistics"""
        return {
            **self.stats,
            'total_samples': len(self),
            'cache_info': self._load_3d_stack_cached.cache_info() if hasattr(self, '_load_3d_stack_cached') else "N/A"
        }

    def clear_cache(self):
        """Memory cleanup"""
        if hasattr(self, '_load_3d_stack_cached'):
            self._load_3d_stack_cached.cache_clear()
        gc.collect()

## 2.5D EfficientNet Model

In [ ]:
!pip install -q timm

import torch.nn as nn
import timm

class LabelQueryCrossAttention(nn.Module):
    def __init__(self, feature_dim=512, num_classes=12):
        super().__init__()
        # Learnable Queries for 12 Target Abnormality Conditions
        self.query_embed = nn.Parameter(torch.randn(num_classes, feature_dim))
        self.mha = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=8, batch_first=True)
        self.fc = nn.Linear(feature_dim, 1)

    def forward(self, x):
        # x shape: (B, Depth, Feature_Dim)
        B = x.shape[0]
        queries = self.query_embed.unsqueeze(0).repeat(B, 1, 1) # (B, 12, 512)

        # Cross-Attention: Query=Targets (12), Key/Value=Slices (32)
        attn_out, _ = self.mha(query=queries, key=x, value=x) # (B, 12, 512)
        logits = self.fc(attn_out).squeeze(-1) # (B, 12)
        return logits

class KneeAbnormalityModel2_5D(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=12, depth=32):
        super().__init__()
        self.depth = depth
        # 2D CNN Feature Extractor
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        feature_dim = self.backbone.num_features

        # Label-Query Cross Attention Module
        self.attention_head = LabelQueryCrossAttention(feature_dim=feature_dim, num_classes=num_classes)

    def forward(self, x):
        # x input: (B, C, Depth, H, W) -> (B, 3, 32, 224, 224)
        B, C, D, H, W = x.shape

        # Reshape to process all slices via 2D CNN
        x = x.permute(0, 2, 1, 3, 4).contiguous().view(B * D, C, H, W)
        features = self.backbone(x) # (B * D, Feature_Dim)

        # Reshape back to sequence of slices: (B, Depth, Feature_Dim)
        features = features.view(B, D, -1)

        # Predict 12 target logits via Attention Pooling
        logits = self.attention_head(features)
        return logits

## Masked Binary Cross-Entropy Loss & Training Loop

In [ ]:
class MaskedBCEWithLogitsLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets, masks):
        raw_loss = self.bce(logits, targets)
        masked_loss = raw_loss * masks
        return masked_loss.sum() / (masks.sum() + 1e-7)

# --- Execution Test ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

model = KneeAbnormalityModel2_5D().to(device)
criterion = MaskedBCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

# Verify single forward-backward pass
train_fold = train_df[train_df['fold'] != 0]
train_ds = KneeAbnormalityDataset(train_fold, train_series_df, DATA_DIR, target_cols=target_cols)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)

batch = next(iter(train_loader))
images = batch['image'].to(device)
labels = batch['label'].to(device)
masks = batch['mask'].to(device)

optimizer.zero_grad()
outputs = model(images)
loss = criterion(outputs, labels, masks)
loss.backward()
optimizer.step()

print(f"\n🎉 Successful End-to-End Training Pass! Calculated Loss: {loss.item():.4f}")

## auc cure etc

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import timm

# ---------------------------------------------------------
# 1. PATHS & GLOBAL SETUP
# ---------------------------------------------------------
DATA_DIR = '/content/drive/MyDrive/Kaggle_Knee_Data'
MODEL_SAVE_DIR = os.path.join(DATA_DIR, 'model_checkpoints')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Training Device: {device}")

# Load Pre-processed Split CSV
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train_folds.csv'))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, 'train_series.csv'))
target_cols = [c for c in train_df.columns if c not in ['StudyInstanceUID', 'fold']]

# ---------------------------------------------------------
# 2. MASKED BCE LOSS FUNCTION
# ---------------------------------------------------------
class MaskedBCEWithLogitsLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets, masks):
        raw_loss = self.bce(logits, targets)
        masked_loss = raw_loss * masks
        return masked_loss.sum() / (masks.sum() + 1e-7)

# ---------------------------------------------------------
# 3. TRAINING & VALIDATION ENGINE FOR 1 EPOCH
# ---------------------------------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch in dataloader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        masks = batch['mask'].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)

@torch.no_grad()
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    all_masks = []

    for batch in dataloader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        masks = batch['mask'].to(device)

        outputs = model(images)
        loss = criterion(outputs, labels, masks)

        running_loss += loss.item()

        # Apply Sigmoid to convert logits into probabilities [0, 1]
        probs = torch.sigmoid(outputs)

        all_preds.append(probs.cpu().numpy())
        all_targets.append(labels.cpu().numpy())
        all_masks.append(masks.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    all_masks = np.vstack(all_masks)

    # Compute Mean ROC-AUC only on valid/unmasked (-1 excluded) targets
    auc_scores = []
    for col_idx in range(all_targets.shape[1]):
        valid_idx = all_masks[:, col_idx] == 1.0
        y_true = all_targets[valid_idx, col_idx]
        y_pred = all_preds[valid_idx, col_idx]

        # Check if both classes (0 and 1) exist in valid targets
        if len(np.unique(y_true)) > 1:
            score = roc_auc_score(y_true, y_pred)
            auc_scores.append(score)

    mean_auc = np.mean(auc_scores) if len(auc_scores) > 0 else 0.5
    return running_loss / len(dataloader), mean_auc, all_preds

# ---------------------------------------------------------
# 4. FULL 5-FOLD TRAINING LOOP (SAVING CHECKPOINTS TO DRIVE)
# ---------------------------------------------------------
EPOCHS = 10
BATCH_SIZE = 4
LR = 3e-4

oof_predictions = np.zeros((len(train_df), len(target_cols)))

print("\n" + "="*50)
print("STARTING 5-FOLD MODEL TRAINING LOOP")
print("="*50)

for fold in range(5):
    print(f"\n⚡ --- Training Fold {fold} --- ⚡")

    # Split Train/Val Data for current fold
    train_data = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_data = train_df[train_df['fold'] == fold].reset_index(drop=True)
    val_indices = train_df[train_df['fold'] == fold].index

    # DataLoaders
    train_ds = KneeAbnormalityDataset(train_data, train_series_df, DATA_DIR, target_cols=target_cols, is_train=True)
    val_ds = KneeAbnormalityDataset(val_data, train_series_df, DATA_DIR, target_cols=target_cols, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Initialize Model, Optimizer, Loss
    model = KneeAbnormalityModel2_5D(model_name='efficientnet_b0', num_classes=len(target_cols)).to(device)
    criterion = MaskedBCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_auc = 0.0
    best_fold_preds = None
    checkpoint_path = os.path.join(MODEL_SAVE_DIR, f'best_model_fold_{fold}.pt')

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_auc, val_preds = validate_one_epoch(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_auc:.4f}")

        # Save Best Checkpoint to Drive
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_fold_preds = val_preds
            torch.save(model.state_dict(), checkpoint_path)
            print(f"   💾 Saved new best model checkpoint to Drive (AUC: {best_val_auc:.4f})")

    # Store Best Out-Of-Fold (OOF) Predictions
    oof_predictions[val_indices] = best_fold_preds
    print(f"✅ Fold {fold} Complete! Best Validation ROC-AUC: {best_val_auc:.4f}")

# Save Out-of-fold predictions to CSV
oof_df = train_df[['StudyInstanceUID', 'fold']].copy()
for i, col in enumerate(target_cols):
    oof_df[f'pred_{col}'] = oof_predictions[:, i]
oof_df.to_csv(os.path.join(DATA_DIR, 'oof_predictions.csv'), index=False)

# ---------------------------------------------------------
# 5. OVERALL OUT-OF-FOLD (OOF) ROC-AUC SCORE & BLENDING
# ---------------------------------------------------------
print("\n" + "="*50)
print("FINAL VALIDATION & BLENDING EVALUATION")
print("="*50)

# Calculate Overall OOF ROC-AUC
overall_auc_scores = []
for i, col in enumerate(target_cols):
    valid_idx = train_df[col].values != -1
    y_true = train_df.loc[valid_idx, col].values
    y_pred = oof_predictions[valid_idx, i]
    score = roc_auc_score(y_true, y_pred)
    overall_auc_scores.append(score)
    print(f"Class: {col:<30} | ROC-AUC: {score:.4f}")

overall_mean_auc = np.mean(overall_auc_scores)
print("-" * 50)
print(f"🏆 OVERALL 5-FOLD OUT-OF-FOLD (OOF) ROC-AUC SCORE: {overall_mean_auc:.4f}")
print("-" * 50)

# ---------------------------------------------------------
# 6. RANK-AVERAGED ENSEMBLING / INFERENCE FUNCTION
# ---------------------------------------------------------
def rank_average_predictions(preds_list):
    """
    Takes a list of prediction arrays [(N, 12), (N, 12), ...] from 5 fold models
    and returns rank-averaged blended predictions.
    """
    ranked_preds = np.zeros_like(preds_list[0])
    num_models = len(preds_list)

    for col in range(preds_list[0].shape[1]):
        col_rank_sum = np.zeros(len(preds_list[0]))
        for model_preds in preds_list:
            col_rank_sum += rankdata(model_preds[:, col]) / len(model_preds)
        ranked_preds[:, col] = col_rank_sum / num_models

    return ranked_preds

print("\n🎉 Model Training, Validation Score Calculation, and Rank Blending Pipeline fully completed!")